<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collecting & Cleaning

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import requests
from google.colab import userdata
from google.colab import drive
from pathlib import Path
from pprint import pprint as print

In [2]:

# mounting drive for future use
drive.mount('drive')

Drive already mounted at drive; to attempt to forcibly remount, call drive.mount("drive", force_remount=True).


## Data Gathering

In [3]:
# the api to query data from
data_api = "https://data.cityofnewyork.us/api/v3/views/irx2-vgm4/query.json"

In [4]:
# Headers and parameters
headers = {
    'X-App-Token': userdata.get('token'),
    "Content-Type": "application/json",
  }
payload = {
    "query":"SELECT * LIMIT 1"
}

In [5]:
# gathering temporary data
response = requests.get(data_api, headers=headers, params=payload)
data = response.json()
print(data)

[{':created_at': '2026-09-29T15:57:26.264Z',
  ':id': 'row-9zdp-t8z2.v88s',
  ':updated_at': '2026-09-29T15:57:26.264Z',
  ':version': 'rv-idqs~t3a8-8mk5',
  'airport_fee': '0.0',
  'cbd_congestion_fee': '0.0',
  'congestion_surcharge': '2.5',
  'dolocationid': '237',
  'extra': '3.5',
  'fare_amount': '10.0',
  'improvement_surcharge': '1.0',
  'mta_tax': '0.5',
  'passenger_count': '1.0',
  'payment_type': '1',
  'pulocationid': '229',
  'ratecodeid': '1.0',
  'store_and_fwd_flag': 'N',
  'tip_amount': '3.0',
  'tolls_amount': '0.0',
  'total_amount': '18.0',
  'tpep_dropoff_datetime': '2025-01-01T00:26:59.000',
  'tpep_pickup_datetime': '2025-01-01T00:18:38.000',
  'trip_distance': '1.6',
  'vendorid': '1'}]


In [6]:
# gathering the right data
query = """
SELECT
  count(*)
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 1
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
query_data = query_response.json()
print(query_data)

[{'count': '11707067'}]


In [7]:
# Our goal is to collect ~11 million rows of data.
query = f"""
SELECT
  *
WHERE
  tpep_pickup_datetime >= "2025-06-21T00:00:00"
  AND
  tpep_dropoff_datetime <= "2025-09-21T23:59:59"
LIMIT 12000000
"""

query_payload = {
    "query": query
}

query_response = requests.get(data_api, headers=headers, params=query_payload)
print("All Done!!!")
query_data = query_response.json()
taxi_df = pd.DataFrame(query_data)

'All Done!!!'


## Data Preview

In [8]:
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
0,2,2025-06-21T00:00:00.000,2025-06-21T00:07:21.000,1.0,1.17,1.0,N,137,113,1,...,1.0,16.71,2.5,0.0,0.75,row-emei~byqv-hzqe,rv-dnr4~ztm9.kep3,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
1,2,2025-06-21T00:00:00.000,2025-06-21T00:04:21.000,2.0,0.61,1.0,N,79,107,1,...,1.0,13.86,2.5,0.0,0.75,row-h2b5-wa6w_t924,rv-qyaa~vq94-iwvb,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
2,2,2025-06-21T00:00:00.000,2025-06-21T00:08:00.000,NaN,1.7,NaN,NaN,74,42,0,...,1.0,2.17,NaN,NaN,0.0,row-xpsy~jicp~dcpm,rv-awpk_x4fp~ci3q,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
3,2,2025-06-21T00:00:00.000,2025-06-21T00:29:45.000,NaN,12.49,NaN,NaN,261,244,0,...,1.0,48.18,NaN,NaN,0.75,row-9xbz-xfrr_zane,rv-efyf~7pu6_7u7f,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN
4,2,2025-06-21T00:00:00.000,2025-06-21T00:11:00.000,NaN,1.73,NaN,NaN,50,162,0,...,1.0,25.21,NaN,NaN,0.75,row-fqhg.5ceh~usg7,rv-xevg~cyge.7u66,2026-09-30T13:45:57.389Z,2026-09-30T13:45:57.389Z,NaN


In [9]:
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
11707062,7,2025-09-21T23:58:08.000,2025-09-21T23:58:08.000,1.0,0.89,1.0,N,186,234,1,...,1.0,14.7,2.5,0.0,0.75,row-29ph~gzpu_tazy,rv-xkhe.6g4x-xzhz,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707063,2,2025-09-21T23:58:15.000,2025-09-21T23:58:20.000,2.0,0.0,5.0,N,265,265,1,...,1.0,67.0,0.0,0.0,0.0,row-4g6k~vzic_7j3y,rv-zkyc-739x~wjtu,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707064,2,2025-09-21T23:58:26.000,2025-09-21T23:58:30.000,1.0,0.05,5.0,N,265,265,1,...,1.0,151.0,0.0,0.0,0.0,row-jig5-i2j5_wxsw,rv-hack~bkni~b43c,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707065,7,2025-09-21T23:58:54.000,2025-09-21T23:58:54.000,1.0,4.73,1.0,N,48,179,1,...,1.0,30.45,2.5,0.0,0.75,row-mt2k.x3q7~ypn4,rv-ca5i-jvv7.n7tt,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN
11707066,7,2025-09-21T23:59:43.000,2025-09-21T23:59:43.000,1.0,1.03,1.0,N,249,107,1,...,1.0,18.81,2.5,0.0,0.75,row-hgds.bskj~drqf,rv-fknd-iyz9~ecyd,2026-09-30T16:34:52.657Z,2026-09-30T16:34:52.657Z,NaN


## Data Structure

In [10]:
print(taxi_df.shape)

(11707067, 25)


In [11]:
print(taxi_df.columns)

Index(['vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag',
       'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee',
       'cbd_congestion_fee', ':id', ':version', ':created_at', ':updated_at',
       'request_source'],
      dtype='object')


In [12]:
taxi_df.dtypes

,0
vendorid,object
tpep_pickup_datetime,object
tpep_dropoff_datetime,object
passenger_count,object
trip_distance,object
ratecodeid,object
store_and_fwd_flag,object
pulocationid,object
dolocationid,object
payment_type,object


## Data Engineering

In [13]:
# missing values
print(taxi_df.isnull().sum())

vendorid                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          3113406
trip_distance                  0
ratecodeid               3113406
store_and_fwd_flag       3113406
pulocationid                   0
dolocationid                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     3113406
airport_fee              3113406
cbd_congestion_fee             0
:id                            0
:version                       0
:created_at                    0
:updated_at                    0
request_source           8744418
dtype: int64


In [14]:
# gathering all numeric columns to operate on
non_numeric_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'store_and_fwd_flag', 'request_source']
numric_columns = [col for col in taxi_df.columns if col not in non_numeric_columns]
print(numric_columns)

['vendorid',
 'passenger_count',
 'trip_distance',
 'ratecodeid',
 'pulocationid',
 'dolocationid',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee',
 'cbd_congestion_fee',
 ':id',
 ':version',
 ':created_at',
 ':updated_at']


In [15]:
# numeric column adjusting
for col in numric_columns:
  taxi_df[col] = pd.to_numeric(taxi_df[col], errors='coerce')

In [16]:
# date column adjusting
taxi_df['tpep_pickup_datetime'] = pd.to_datetime(taxi_df['tpep_pickup_datetime'], format="mixed")
taxi_df['tpep_dropoff_datetime'] = pd.to_datetime(taxi_df['tpep_dropoff_datetime'], format="mixed")

In [17]:
taxi_df.head()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
0,2,2025-06-21,2025-06-21 00:07:21,1.0,1.17,1.0,N,137,113,1,...,1.0,16.71,2.5,0.0,0.75,NaN,NaN,NaN,NaN,NaN
1,2,2025-06-21,2025-06-21 00:04:21,2.0,0.61,1.0,N,79,107,1,...,1.0,13.86,2.5,0.0,0.75,NaN,NaN,NaN,NaN,NaN
2,2,2025-06-21,2025-06-21 00:08:00,NaN,1.70,NaN,NaN,74,42,0,...,1.0,2.17,NaN,NaN,0.00,NaN,NaN,NaN,NaN,NaN
3,2,2025-06-21,2025-06-21 00:29:45,NaN,12.49,NaN,NaN,261,244,0,...,1.0,48.18,NaN,NaN,0.75,NaN,NaN,NaN,NaN,NaN
4,2,2025-06-21,2025-06-21 00:11:00,NaN,1.73,NaN,NaN,50,162,0,...,1.0,25.21,NaN,NaN,0.75,NaN,NaN,NaN,NaN,NaN


In [18]:
taxi_df.tail()

,vendorid,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,pulocationid,dolocationid,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,:id,:version,:created_at,:updated_at,request_source
11707062,7,2025-09-21 23:58:08,2025-09-21 23:58:08,1.0,0.89,1.0,N,186,234,1,...,1.0,14.70,2.5,0.0,0.75,NaN,NaN,NaN,NaN,NaN
11707063,2,2025-09-21 23:58:15,2025-09-21 23:58:20,2.0,0.00,5.0,N,265,265,1,...,1.0,67.00,0.0,0.0,0.00,NaN,NaN,NaN,NaN,NaN
11707064,2,2025-09-21 23:58:26,2025-09-21 23:58:30,1.0,0.05,5.0,N,265,265,1,...,1.0,151.00,0.0,0.0,0.00,NaN,NaN,NaN,NaN,NaN
11707065,7,2025-09-21 23:58:54,2025-09-21 23:58:54,1.0,4.73,1.0,N,48,179,1,...,1.0,30.45,2.5,0.0,0.75,NaN,NaN,NaN,NaN,NaN
11707066,7,2025-09-21 23:59:43,2025-09-21 23:59:43,1.0,1.03,1.0,N,249,107,1,...,1.0,18.81,2.5,0.0,0.75,NaN,NaN,NaN,NaN,NaN


## Duplicates

In [19]:
taxi_df.shape

(11707067, 25)

In [20]:
taxi_df = taxi_df.drop_duplicates().reset_index(drop=True)

In [21]:
taxi_df.shape

(11707066, 25)

## Missing Values

In [22]:
# calculate percent of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

,0
vendorid,0.000000
tpep_pickup_datetime,0.000000
tpep_dropoff_datetime,0.000000
passenger_count,26.594247
trip_distance,0.000000
ratecodeid,26.594247
store_and_fwd_flag,26.594247
pulocationid,0.000000
dolocationid,0.000000
payment_type,0.000000


In [23]:
# ratecode value counts
"""
Possible values

1= Standard rate
2=JFK
3=Newark
4=Nassau or Westchester
5=Negotiated fare
6=Group ride
99 = Null/unknown

"""
print(taxi_df['ratecodeid'].value_counts(dropna=False))
print(taxi_df['ratecodeid'].dtype)

ratecodeid
1.0     8059781
NaN     3113406
2.0      329229
5.0      126872
3.0       41044
4.0       34420
99.0       2306
6.0           8
Name: count, dtype: int64
dtype('float64')


In [24]:
# replacing missing ratecode values
taxi_df['ratecodeid'].fillna(99, inplace=True)

/tmp/ipykernel_4511/2785377976.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['ratecodeid'].fillna(99, inplace=True)


In [25]:
print(taxi_df['ratecodeid'].value_counts(dropna=False))

ratecodeid
1.0     8059781
99.0    3115712
2.0      329229
5.0      126872
3.0       41044
4.0       34420
6.0           8
Name: count, dtype: int64


In [26]:
# store_and_fwd_flag value counts
"""
Possible Values

Y= store and forward trip
N= not a store and forward trip

"""
print(taxi_df['store_and_fwd_flag'].value_counts(dropna=False))

store_and_fwd_flag
N      8568174
NaN    3113406
Y        25486
Name: count, dtype: int64


In [27]:
# store and forward serves only a database usefullness, so it is not needed here
taxi_df.drop(columns=['store_and_fwd_flag'], axis=1, inplace=True)

In [28]:
  # congestion_surcharge value counts
print(taxi_df['congestion_surcharge'].value_counts(dropna=False))

congestion_surcharge
 2.50    7687339
 NaN     3113406
 0.00     727294
-2.50     179021
 1.00          5
 0.75          1
Name: count, dtype: int64


In [29]:
# replace values
taxi_df['congestion_surcharge'].replace(
    {
      2.50: 1,
      0.00: 0,
      -2.50: 1,
      1.00: 1,
      0.75: 1
    },
    inplace=True
)

taxi_df['congestion_surcharge'].fillna(-1, inplace=True)

/tmp/ipykernel_4511/3545522270.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['congestion_surcharge'].replace(
/tmp/ipykernel_4511/3545522270.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using '

In [30]:
# airport_fee value counts
print(taxi_df['airport_fee'].value_counts(dropna=False))

airport_fee
 0.00    7700775
 NaN     3113406
 1.75     834199
-1.75      52905
 6.75       4535
 5.00       1245
 0.50          1
Name: count, dtype: int64


In [31]:
# replace values
taxi_df['airport_fee'].replace(
    {
      0.00: 0,
      1.75: 1,
      -1.75: 1,
      6.75: 1,
      5.00: 1,
      0.50: 1
    },
    inplace=True
)

taxi_df['airport_fee'].fillna(-1, inplace=True)

/tmp/ipykernel_4511/3638072713.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  taxi_df['airport_fee'].replace(
/tmp/ipykernel_4511/3638072713.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method

In [32]:
# request_source value counts
"""
Possible Values

Origin of e-hail request. ID's can be provided for forwarded trips:

"CC": Call center (required for accessible dispatch),

"A": Company's own app,

"HVXXXX": High volume ID,

"EHXXXX": Ehail ID

"""
print(taxi_df['request_source'].value_counts(dropna=False))

request_source
NaN       8744417
HV0003    2703632
A          182336
EH0010      53209
EH0004      23255
CC            217
Name: count, dtype: int64


In [33]:
# since request source has a large portion of missing values, and the focus is borough, not call center, it should be removed
taxi_df.drop(columns=['request_source'], axis=1, inplace=True)

In [34]:
# calculate percent of missing data in each column
percent_missing = (taxi_df.isna().sum() / len(taxi_df)) * 100
percent_missing

,0
vendorid,0.000000
tpep_pickup_datetime,0.000000
tpep_dropoff_datetime,0.000000
passenger_count,26.594247
trip_distance,0.000000
ratecodeid,0.000000
pulocationid,0.000000
dolocationid,0.000000
payment_type,0.000000
fare_amount,0.000000


## Taxi Zones Data

In [35]:
# gathering temporary data
zones_query = {
    "query":"""
    SELECT
      *
    LIMIT 500
    """
}
zones_response = requests.get(
    "https://data.cityofnewyork.us/api/v3/views/8meu-9t5y/query.json",
    headers=headers,
    params=zones_query
  )
zones_data = zones_response.json()
print("All done!")

'All done!'


In [36]:
zone_df = pd.DataFrame(zones_data)

In [37]:
zone_df.head()

,the_geom,shape_leng,shape_area,zone,locationid,borough,:id,:version,:created_at,:updated_at
0,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",0.116357453189,0.0007823067885,Newark Airport,1,EWR,row-rgxy_42jk_dq7g,rv-a7vs.vwuy~tyci,2025-06-09T17:12:22.062Z,2025-06-09T17:12:22.062Z
1,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",0.43346966679,0.00486634037837,Jamaica Bay,2,Queens,row-76qp.92qd~k3iu,rv-2qzz~i35d~7f8b,2025-06-09T17:12:22.062Z,2025-06-09T17:12:22.062Z
2,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",0.0843411059012,0.000314414156821,Allerton/Pelham Gardens,3,Bronx,row-j55v~b782-bey7,rv-65ty-i8fr~zrr5,2025-06-09T17:12:22.062Z,2025-06-09T17:12:22.062Z
3,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",0.0435665270921,0.000111871946192,Alphabet City,4,Manhattan,row-pnpa-ap69~6m7d,rv-wvih~xy8h~tpys,2025-06-09T17:12:22.062Z,2025-06-09T17:12:22.062Z
4,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",0.0921464898574,0.000497957489363,Arden Heights,5,Staten Island,row-ewiw.9man~jjqn,rv-t95v.ubu4-6y4b,2025-06-09T17:12:22.062Z,2025-06-09T17:12:22.062Z


In [38]:
zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   the_geom     263 non-null    object
 1   shape_leng   263 non-null    object
 2   shape_area   263 non-null    object
 3   zone         263 non-null    object
 4   locationid   263 non-null    object
 5   borough      263 non-null    object
 6   :id          263 non-null    object
 7   :version     263 non-null    object
 8   :created_at  263 non-null    object
 9   :updated_at  263 non-null    object
dtypes: object(10)
memory usage: 20.7+ KB


In [39]:
zone_df.drop(columns=[':id', ':version', ':created_at', ':updated_at'], axis=1, inplace=True)

In [40]:
zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   the_geom    263 non-null    object
 1   shape_leng  263 non-null    object
 2   shape_area  263 non-null    object
 3   zone        263 non-null    object
 4   locationid  263 non-null    object
 5   borough     263 non-null    object
dtypes: object(6)
memory usage: 12.5+ KB


In [41]:
zone_df['shape_leng'] = zone_df['shape_leng'].astype("float")

In [42]:
zone_df['shape_area'] = zone_df['shape_area'].astype("float")

In [43]:
zone_df['locationid'] = zone_df['locationid'].astype("int")

In [44]:
zone_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   the_geom    263 non-null    object 
 1   shape_leng  263 non-null    float64
 2   shape_area  263 non-null    float64
 3   zone        263 non-null    object 
 4   locationid  263 non-null    int64  
 5   borough     263 non-null    object 
dtypes: float64(2), int64(1), object(3)
memory usage: 12.5+ KB


## Feature Creation

In [45]:
print(taxi_df.columns)
print(zone_df.columns)

Index(['vendorid', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'ratecodeid', 'pulocationid',
       'dolocationid', 'payment_type', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee', ':id',
       ':version', ':created_at', ':updated_at'],
      dtype='object')
Index(['the_geom', 'shape_leng', 'shape_area', 'zone', 'locationid',
       'borough'],
      dtype='object')


In [46]:
taxi_df.drop(columns=['vendorid', ':id',':version', ':created_at', ':updated_at'], inplace=True)

In [53]:
pu_data = taxi_df['pulocationid'].value_counts().reset_index()
pu_data.columns = ['locationid', 'count']
do_data = taxi_df['dolocationid'].value_counts().reset_index()
do_data.columns = ['locationid', 'count']
print(pu_data.head())
print(do_data.head())

   locationid   count
0         132  571572
1         161  502232
2         237  460873
3         186  389987
4         236  383018
   locationid   count
0         161  419155
1         237  415743
2         236  400336
3         230  363872
4         170  331179


In [55]:
# combining pu and drop off zones to keep only zones and zone counts
pu_do_data = pu_data.merge(do_data, on='locationid', how='outer')
pu_do_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 262 entries, 0 to 261
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   locationid  262 non-null    int64
 1   count_x     262 non-null    int64
 2   count_y     262 non-null    int64
dtypes: int64(3)
memory usage: 6.3 KB


In [56]:
# combining location id and location names from zone_df
pu_do_data = pu_do_data.merge(zone_df, on='locationid', how='left')
pu_do_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   locationid  263 non-null    int64  
 1   count_x     263 non-null    int64  
 2   count_y     263 non-null    int64  
 3   the_geom    259 non-null    object 
 4   shape_leng  259 non-null    float64
 5   shape_area  259 non-null    float64
 6   zone        259 non-null    object 
 7   borough     259 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 16.6+ KB


In [57]:
pu_do_data['borough'].value_counts()

,count
borough,
Queens,69
Manhattan,66
Brooklyn,61
Bronx,43
Staten Island,19
EWR,1


In [58]:
pu_do_data['zone'].value_counts()

,count
zone,
Corona,2
Mount Hope,1
Murray Hill,1
Murray Hill-Queens,1
New Dorp/Midland Beach,1
...,...
Far Rockaway,1
Erasmus,1
Eltingville/Annadale/Prince's Bay,1


## Export

In [60]:
taxi_df.to_csv("/content/drive/MyDrive/taxi_summer_data.csv", index=False)
zone_df.to_json("/content/drive/MyDrive/taxi_zones_data.json", index=False)